# LogicTune quickstart (CPU only)

Parse a controller, model check it against the 15 LTL specifications, and read the recorded results.
No GPU and no model download are needed. Run from the repository root (`jupyter lab notebooks/quickstart.ipynb`).

In [ ]:
import sys, json, os
sys.path.insert(0, os.path.abspath("../src") if os.path.basename(os.getcwd()) == "notebooks" else "src")
from logictune import SYSTEM_PROMPT, extract_rules, verify_response, format_result, find_nuxmv
print(SYSTEM_PROMPT)
print("nuXmv:", find_nuxmv())

## 1. Parse and verify a response

In [ ]:
response = """1. If pedestrian_in_front, stop.
2. If stop_sign, stop.
3. If not green_traffic_light, stop.
4. If opposite_car or car_from_left or car_from_right or pedestrian_at_left or pedestrian_at_right, stop.
5. Otherwise, go_straight."""
engine = "both" if find_nuxmv() else "explicit"
print(format_result(verify_response(response, engine=engine)))

## 2. Rule order matters

In [ ]:
bad = "1. If the light is green, go straight.\n2. If a pedestrian is in front, stop."
res = verify_response(bad)
print(res.score, "/ 15 ->", [k for k, v in res.specs.items() if not v])

## 3. Read the recorded results

In [ ]:
root = "../results/latest" if os.path.basename(os.getcwd()) == "notebooks" else "results/latest"
index = json.load(open(f"{root}/index.json"))
for run in index["runs"]:
    s = json.load(open(f"{root}/{run['path']}summary.json"))
    for r in s["rounds"]:
        lo, hi = r["satisfaction_ci95"]
        print(f"{run['id']:>14}  round {r['round']}: {r['satisfaction_rate']:.1%}  (95% CI {lo:.1%}-{hi:.1%})  full pass {r['full_pass_rate']:.1%}")

## 4. Replay one recorded controller

In [ ]:
run = index["runs"][0]
replay = json.load(open(f"{root}/{run['path']}replay.json"))
final = replay["models"][-1]["id"]
sample = next(x for x in replay["samples"][final] if x["scenario_id"] == 1)
print(sample["response"][:800])
print(format_result(verify_response(sample["response"])))